# Cuaderno 2. Recuperación e inferencia con los pipelines guardados

**Curso:** Machine Learning | **Integrante:** Jaime Alejandro Ocampo Quiceno

**Qué hace este cuaderno:** carga `pipeline_prep.joblib` y `pipeline_modelo.joblib` y demuestra que dan las mismas métricas que el Cuaderno 1. **No se usa `fit` ni `fit_transform`.**

In [1]:
# Todas las librerías se importan aquí, al inicio
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import sklearn, scipy, joblib
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

SEMILLA = 42
RAIZ = Path("..")
RUTA_DATOS = RAIZ / "data" / "saber11_eda.csv"
RUTA_ARTEFACTOS = RAIZ / "artifacts"
RUTA_RESULTADOS = RAIZ / "resultados"

# Comprobación del kernel y de las versiones
print("Python ejecutable:", sys.executable)
print("Python:", sys.version.split()[0], "| numpy:", np.__version__, "| pandas:", pd.__version__)
print("scipy:", scipy.__version__, "| scikit-learn:", sklearn.__version__, "| joblib:", joblib.__version__)

# Versiones con las que se entrenó (guardadas por el Cuaderno 1)
with open(RUTA_RESULTADOS / "metricas_cuaderno1.json") as f:
    ref = json.load(f)
print("Versiones del entrenamiento:", ref["versiones"])
assert ref["versiones"]["sklearn"] == sklearn.__version__, "Versión de scikit-learn distinta a la del entrenamiento"

Python ejecutable: c:\Users\aleh\Desktop\laboratorio_uv_ml\venv_ml\Scripts\python.exe
Python: 3.12.16 | numpy: 2.5.5 | pandas: 3.1.1
scipy: 1.18.1 | scikit-learn: 1.9.1 | joblib: 1.6.0
Versiones del entrenamiento: {'python': '3.12.16', 'numpy': '2.5.5', 'scipy': '1.18.1', 'sklearn': '1.9.1', 'joblib': '1.6.0'}


**Análisis.** Se verifica que el kernel sea `Python (venv_ml)` y que la versión de scikit-learn sea igual a la del entrenamiento. Cargar un `.joblib` con otra versión puede dar errores o resultados distintos.

## 1. Cargar los pipelines

In [2]:
pipeline_prep = joblib.load(RUTA_ARTEFACTOS / "pipeline_prep.joblib")
pipeline_modelo = joblib.load(RUTA_ARTEFACTOS / "pipeline_modelo.joblib")
print(type(pipeline_prep).__name__, "|", type(pipeline_modelo).__name__)

Pipeline | TransformedTargetRegressor


## 2. Reconstruir la misma partición de prueba

Se repiten los mismos pasos del Cuaderno 1 (misma limpieza, misma muestra y misma semilla) para obtener exactamente los mismos estudiantes de prueba. Aquí solo se usa la prueba.

In [3]:
df = pd.read_csv(RUTA_DATOS)
df = df[df["PUNT_GLOBAL"] > 0].copy()
df = df.sample(6000, random_state=SEMILLA).reset_index(drop=True)

VARS_ENTRADA = ["ESTU_INSE_INDIVIDUAL", "FAMI_TIENEINTERNET", "FAMI_TIENECOMPUTADOR",
                "COLE_NATURALEZA", "COLE_AREA_UBICACION"]
X = df[VARS_ENTRADA]
y = df["PUNT_GLOBAL"]
_, X_pru, _, y_pru = train_test_split(X, y, test_size=0.25, random_state=SEMILLA)
print("Estudiantes de prueba:", X_pru.shape)

Estudiantes de prueba: (1500, 5)


## 3. Transformar, predecir y evaluar (sin reentrenar)

In [4]:
X_pru_prep = pipeline_prep.transform(X_pru)          # solo transform
y_pred = pipeline_modelo.predict(X_pru_prep)         # solo predict

rmse = float(np.sqrt(mean_squared_error(y_pru, y_pred)))
mae = float(mean_absolute_error(y_pru, y_pred))
r2 = float(r2_score(y_pru, y_pred))
print(f"Cuaderno 2  -> RMSE = {rmse:.3f} | MAE = {mae:.3f} | R2 = {r2:.3f}")
print(f"Cuaderno 1  -> RMSE = {ref['rmse']:.3f} | MAE = {ref['mae']:.3f} | R2 = {ref['r2']:.3f}")

# Comprobación: mismas métricas y mismas predicciones que en el entrenamiento
pred_ref = np.load(RUTA_RESULTADOS / "pred_prueba_cuaderno1.npy")
assert np.isclose(rmse, ref["rmse"]) and np.isclose(mae, ref["mae"]) and np.isclose(r2, ref["r2"])
assert np.allclose(y_pred, pred_ref)
print("OK: las métricas y las predicciones son iguales a las del Cuaderno 1.")

Cuaderno 2  -> RMSE = 43.878 | MAE = 35.336 | R2 = 0.204
Cuaderno 1  -> RMSE = 43.878 | MAE = 35.336 | R2 = 0.204
OK: las métricas y las predicciones son iguales a las del Cuaderno 1.


**Análisis.** Las métricas y las 1.500 predicciones son iguales a las del Cuaderno 1. Esto demuestra que los archivos guardados conservan lo aprendido y que se puede predecir sin reentrenar.

## 4. Predicción para un lote pequeño de estudiantes nuevos

Se simulan cinco estudiantes nuevos. El último tiene datos vacíos a propósito para comprobar que el pipeline los rellena con lo aprendido en el entrenamiento.

In [5]:
nuevos = pd.DataFrame({
    "ESTU_INSE_INDIVIDUAL": [35.0, 48.0, 55.0, 70.0, np.nan],
    "FAMI_TIENEINTERNET":   ["No", "Si", "Si", "Si", None],
    "FAMI_TIENECOMPUTADOR": ["No", "No", "Si", "Si", None],
    "COLE_NATURALEZA":      ["OFICIAL", "OFICIAL", "OFICIAL", "NO OFICIAL", "OFICIAL"],
    "COLE_AREA_UBICACION":  ["RURAL", "URBANO", "URBANO", "URBANO", "URBANO"],
})
nuevos["puntaje_predicho"] = pipeline_modelo.predict(pipeline_prep.transform(nuevos)).round(1)
nuevos

,ESTU_INSE_INDIVIDUAL,FAMI_TIENEINTERNET,FAMI_TIENECOMPUTADOR,COLE_NATURALEZA,COLE_AREA_UBICACION,puntaje_predicho
0,35.0,No,No,OFICIAL,RURAL,215.3
1,48.0,Si,No,OFICIAL,URBANO,242.3
2,55.0,Si,Si,OFICIAL,URBANO,258.3
3,70.0,Si,Si,NO OFICIAL,URBANO,303.8
4,NaN,NaN,NaN,OFICIAL,URBANO,248.1


**Análisis.** El estudiante rural, de colegio oficial y sin internet ni computador recibe el puntaje esperado más bajo, y el de colegio no oficial con INSE alto recibe el más alto. Eso es coherente con lo que mostró el EDA. El pipeline también predijo para el estudiante con datos vacíos, porque usó la mediana y la moda del entrenamiento.

## 5. Preguntas de análisis

**1. ¿Qué diferencia existe entre instalar una librería globalmente y hacerlo dentro de `venv_ml`?**
Globalmente, la librería queda para todo el computador y distintos proyectos pueden chocar por versiones. Dentro de `venv_ml` queda aislada en la carpeta del proyecto, así este laboratorio siempre usa las mismas versiones sin dañar otros trabajos.

**2. ¿Qué ventajas ofrece `uv` frente a instalar paquetes uno por uno con pip?**
`uv` es mucho más rápido, crea entornos con un comando, instala todo desde `requirements.txt` de una vez, resuelve las versiones compatibles entre sí y puede instalar la versión de Python que se necesita.

**3. ¿Qué diferencia hay entre `requirements.txt` y `requirements-lock.txt`?**
`requirements.txt` dice qué librerías se necesitan y en qué rangos de versión (por ejemplo `scikit-learn>=1.6,<2.0`). `requirements-lock.txt` guarda las versiones exactas que se instalaron (por ejemplo `scikit-learn==1.9.1`), así otra persona puede reconstruir el mismo entorno.

**4. ¿Por qué no es suficiente guardar únicamente el clasificador (el modelo)?**
Porque el modelo recibe datos ya preparados. Si no se guarda también el preprocesamiento (medianas, modas, columnas del one hot, escalas), los datos nuevos se prepararían distinto y las predicciones saldrían mal o darían error.

**5. ¿Qué información aprende el `StandardScaler` y qué información aprende PCA, y por qué se incluyen dentro del pipeline de modelamiento?**
El `StandardScaler` aprende la media y la desviación estándar de cada columna. PCA aprende los componentes (las direcciones de mayor variación) y cuánta varianza explica cada uno. En este proyecto se usó el `StandardScaler` y se dejó por fuera PCA porque solo hay 9 entradas. Ambos van dentro del pipeline para que se ajusten solo con entrenamiento y para que los mismos valores se apliquen a datos nuevos.

**6. ¿Qué es la fuga de información y cómo se evita en este laboratorio?**
Es cuando el modelo usa, sin querer, información de los datos de prueba o de la respuesta durante el entrenamiento, y por eso parece mejor de lo que es. Se evita separando entrenamiento y prueba antes de todo, ajustando (`fit`) los dos pipelines solo con entrenamiento, usando solo `transform` y `predict` con la prueba, y sin usar los puntajes por materia como entradas.

**7. ¿Qué riesgos de seguridad tiene cargar archivos `.joblib` de fuentes desconocidas?**
`joblib` usa `pickle`, que puede ejecutar código al cargar el archivo. Un archivo malicioso podría borrar archivos o robar datos del computador. Por eso solo se cargan archivos propios o de fuentes de confianza.

**8. ¿Qué podría ocurrir si los artefactos se cargan con versiones incompatibles de scikit-learn o NumPy?**
Puede dar errores al cargar, advertencias, o peor, predicciones distintas sin avisar, porque la estructura interna de los objetos cambia entre versiones. Por eso se guardan las versiones y se compara la de scikit-learn con un `assert`.